# Volume Collapse Pipeline - Parameter Sensitivity Test

This notebook runs a **one-factor-at-a-time sensitivity test** for the same pipeline used in `volume_collapse_inference.ipynb`:

1. Load data
2. Holdout split
3. Walk-forward validation on optimization period
4. True out-of-sample holdout walk-forward evaluation

It can optionally use the **best trial** from the latest Optuna output in `optimization/`.


## Folder Structure (Observed)

- `volume_collapse_inference.ipynb`: inference pipeline baseline
- `optimization/optimize_volume_collapse.py`: Optuna optimization workflow
- `strategies/volume_collapse_strategy.py`: strategy + config
- `backtesting/`: backtest and walk-forward utilities
- `data/processed/indian_sector_data_2013_2025.csv`: dataset


In [ ]:
from dataclasses import asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from strategies import VolumeCollapseStrategy, VolumeCollapseConfig
from backtesting import Backtester, WalkForwardValidator, WalkForwardConfig


In [ ]:
# -----------------------------------------------------------------------------
# Shared pipeline settings (aligned to volume_collapse_inference.ipynb)
# -----------------------------------------------------------------------------
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for base in candidates:
        if (base / "data/processed/indian_sector_data_2013_2025.csv").exists() and (base / "optimization").exists():
            return base.resolve()
    raise FileNotFoundError(
        f"Could not locate project root from cwd={Path.cwd()}. Expected data/ and optimization/ folders."
    )


PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "data/processed/indian_sector_data_2013_2025.csv"
HOLDOUT_YEARS = 2.0
RISK_FREE_RATE = 0.05

WF_TRAIN_WINDOW = 504
WF_TEST_WINDOW = 63

# Use latest Optuna run for base config if available
USE_LATEST_OPTUNA_BASE = True
OPTUNA_GLOB = str(PROJECT_ROOT / "optimization/volume_collapse_optimization_*.csv")
OUTPUT_DIR = PROJECT_ROOT / "optimization"

print("Pipeline settings loaded")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")


In [ ]:
def load_data(csv_path: str):
    # Load prices and benchmark (same logic as optimization script)
    df = pd.read_csv(csv_path, parse_dates=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce").ffill().dropna(how="all")

    if "Benchmark" in df.columns:
        benchmark = df["Benchmark"]
        prices = df.drop(columns=["Benchmark"])
    else:
        prices = df
        rets = prices.pct_change().fillna(0)
        benchmark = (1 + rets.mean(axis=1)).cumprod()
        benchmark.name = "Benchmark"

    return prices, benchmark


def split_holdout(prices: pd.DataFrame, benchmark: pd.Series, holdout_years: float = 2.0):
    holdout_days = int(holdout_years * 252)
    cutoff_idx = len(prices) - holdout_days

    prices_opt = prices.iloc[:cutoff_idx]
    benchmark_opt = benchmark.iloc[:cutoff_idx]
    prices_holdout = prices.iloc[cutoff_idx:]
    benchmark_holdout = benchmark.iloc[cutoff_idx:]

    return prices_opt, benchmark_opt, prices_holdout, benchmark_holdout


prices_full, benchmark_full = load_data(DATA_PATH)
prices_opt, benchmark_opt, prices_holdout, benchmark_holdout = split_holdout(
    prices_full, benchmark_full, HOLDOUT_YEARS
)

print(f"Full data:  {prices_full.index[0].date()} -> {prices_full.index[-1].date()} ({len(prices_full)} days)")
print(f"Opt data:   {prices_opt.index[0].date()} -> {prices_opt.index[-1].date()} ({len(prices_opt)} days)")
print(f"Holdout:    {prices_holdout.index[0].date()} -> {prices_holdout.index[-1].date()} ({len(prices_holdout)} days)")


In [ ]:
# Baseline config fallback (same values as inference notebook)
default_base_config = VolumeCollapseConfig(
    top_n_sectors=4,
    max_sector_weight=0.30,
    min_sector_weight=0.01,
    rs_lookback=50,
    momentum_lookback=25,
    volatility_window=34,
    use_trend_filter=True,
    trend_ma_period=50,
    weight_smoothing_alpha=0.4,
    full_allocation=True,
    rebalance_frequency="W-FRI",
    vol_window=30,
    vol_percentile=0.20,
    risk_reduction_factor=0.4,
    min_exposure=0.2,
    smooth_scaling=True,
)


def parse_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y", "t"}
    return bool(value)


def load_optuna_best_config(optuna_glob: str):
    files = sorted(Path().glob(optuna_glob), key=lambda p: p.stat().st_mtime)
    if not files:
        return None, None

    latest_file = files[-1]
    df = pd.read_csv(latest_file)
    if "value" not in df.columns:
        return None, None

    best_row = df.loc[df["value"].idxmax()]

    params = {
        "top_n_sectors": int(best_row["params_top_n_sectors"]),
        "max_sector_weight": float(best_row["params_max_sector_weight"]),
        "min_sector_weight": float(best_row["params_min_sector_weight"]),
        "rs_lookback": int(best_row["params_rs_lookback"]),
        "momentum_lookback": int(best_row["params_momentum_lookback"]),
        "volatility_window": int(best_row["params_volatility_window"]),
        "use_trend_filter": parse_bool(best_row["params_use_trend_filter"]),
        "trend_ma_period": int(best_row["params_trend_ma_period"]),
        "weight_smoothing_alpha": float(best_row["params_weight_smoothing_alpha"]),
        "full_allocation": True,
        "rebalance_frequency": "W-FRI",
        "vol_window": int(best_row["params_vol_window"]),
        "vol_percentile": float(best_row["params_vol_percentile"]),
        "risk_reduction_factor": float(best_row["params_risk_reduction_factor"]),
        "min_exposure": float(best_row["params_min_exposure"]),
        "smooth_scaling": parse_bool(best_row["params_smooth_scaling"]),
    }

    return VolumeCollapseConfig(**params), latest_file


base_config = default_base_config
base_source = "inference notebook defaults"

if USE_LATEST_OPTUNA_BASE:
    optuna_config, optuna_file = load_optuna_best_config(OPTUNA_GLOB)
    if optuna_config is not None:
        base_config = optuna_config
        base_source = f"latest optuna best trial: {optuna_file}"

print("Baseline source:", base_source)
print(base_config)


In [ ]:
def evaluate_config(config: VolumeCollapseConfig):
    # Run the same evaluation pipeline used for inference/optimization
    wf_config = WalkForwardConfig(
        train_window=WF_TRAIN_WINDOW,
        test_window=WF_TEST_WINDOW,
        step_size=WF_TEST_WINDOW,
    )

    # Walk-forward on optimization period only
    strategy_wf = VolumeCollapseStrategy(config)
    validator_opt = WalkForwardValidator(strategy_wf, config=wf_config, risk_free_rate=RISK_FREE_RATE)
    wf_result = validator_opt.validate(prices_opt, benchmark_opt)

    # Walk-forward on full timeline, then evaluate only holdout slice
    validator_full = WalkForwardValidator(
        VolumeCollapseStrategy(config),
        config=wf_config,
        risk_free_rate=RISK_FREE_RATE,
    )
    wf_full = validator_full.validate(prices_full, benchmark_full)

    holdout_returns = wf_full.oos_returns.loc[wf_full.oos_returns.index >= prices_holdout.index[0]]
    holdout_returns = holdout_returns[~holdout_returns.index.duplicated(keep="first")].sort_index()

    holdout_bench_returns = benchmark_full.pct_change().loc[holdout_returns.index].fillna(0)
    holdout_equity = 100 * (1 + holdout_returns).cumprod()
    holdout_bench_equity = 100 * (1 + holdout_bench_returns).cumprod()

    metric_engine = Backtester(VolumeCollapseStrategy(config), risk_free_rate=RISK_FREE_RATE)
    holdout_metrics = metric_engine._calculate_metrics(
        holdout_returns,
        holdout_bench_returns,
        holdout_equity,
        holdout_bench_equity,
    )

    return {
        "wf_sharpe": wf_result.aggregate_metrics.get("sharpe_ratio", np.nan),
        "wf_sortino": wf_result.aggregate_metrics.get("sortino_ratio", np.nan),
        "wf_calmar": wf_result.aggregate_metrics.get("calmar_ratio", np.nan),
        "wf_total_return": wf_result.aggregate_metrics.get("total_return", np.nan),
        "holdout_sharpe": holdout_metrics.get("sharpe_ratio", np.nan),
        "holdout_sortino": holdout_metrics.get("sortino_ratio", np.nan),
        "holdout_calmar": holdout_metrics.get("calmar_ratio", np.nan),
        "holdout_total_return": holdout_metrics.get("total_return", np.nan),
        "holdout_max_drawdown": holdout_metrics.get("max_drawdown", np.nan),
    }


baseline_metrics = evaluate_config(base_config)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    if "return" in k or "drawdown" in k:
        print(f"  {k:<22}: {v:>8.2%}")
    else:
        print(f"  {k:<22}: {v:>8.4f}")


## One-Factor-at-a-Time Sensitivity Grid

These values are aligned with the Optuna search ranges in `optimization/optimize_volume_collapse.py`.


In [ ]:
# Keep this moderate; each row runs full WF + holdout evaluation
sensitivity_grid = {
    "top_n_sectors": [2, 3, 4, 5, 6],
    "max_sector_weight": [0.25, 0.30, 0.35, 0.40, 0.50],
    "rs_lookback": [20, 40, 60, 80, 100],
    "momentum_lookback": [5, 10, 15, 20, 25, 30],
    "volatility_window": [10, 22, 34, 46, 52],
    "weight_smoothing_alpha": [0.1, 0.2, 0.3, 0.4, 0.5],
    "vol_window": [30, 40, 50, 60, 70, 80, 100],
    "vol_percentile": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    "risk_reduction_factor": [0.3, 0.4, 0.5, 0.6, 0.7],
    "min_exposure": [0.1, 0.2, 0.3, 0.4],
}

print(f"Total runs planned: {sum(len(v) for v in sensitivity_grid.values())}")


In [ ]:
base_kwargs = asdict(base_config)
records = []

for param, values in sensitivity_grid.items():
    print(f"Testing {param} ({len(values)} values)...")
    for val in values:
        cfg_kwargs = dict(base_kwargs)
        cfg_kwargs[param] = val
        test_config = VolumeCollapseConfig(**cfg_kwargs)

        metrics = evaluate_config(test_config)
        records.append({
            "parameter": param,
            "value": val,
            **metrics,
        })

sensitivity_df = pd.DataFrame(records)
print(f"Completed runs: {len(sensitivity_df)}")


In [ ]:
# Rank results by holdout sharpe and walk-forward sharpe
summary_cols = [
    "parameter",
    "value",
    "wf_sharpe",
    "holdout_sharpe",
    "wf_total_return",
    "holdout_total_return",
    "holdout_max_drawdown",
]

print("Top 20 by holdout_sharpe:")
display(sensitivity_df.sort_values("holdout_sharpe", ascending=False)[summary_cols].head(20))

print("Top 20 by wf_sharpe:")
display(sensitivity_df.sort_values("wf_sharpe", ascending=False)[summary_cols].head(20))


In [ ]:
# Plot per-parameter response curves
params = list(sensitivity_grid.keys())
n = len(params)
cols = 2
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
axes = np.array(axes).reshape(-1)

for i, param in enumerate(params):
    ax = axes[i]
    sub = sensitivity_df[sensitivity_df["parameter"] == param].copy()
    sub = sub.sort_values("value")

    ax.plot(sub["value"], sub["wf_sharpe"], marker="o", label="WF Sharpe")
    ax.plot(sub["value"], sub["holdout_sharpe"], marker="s", label="Holdout Sharpe")

    base_val = base_kwargs[param]
    ax.axvline(base_val, color="gray", linestyle=":", linewidth=1)

    ax.set_title(param)
    ax.set_xlabel("value")
    ax.set_ylabel("Sharpe")
    ax.grid(alpha=0.3)
    ax.legend()

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Sensitivity score: range of metric across the tested values
robustness_rows = []
for param in sensitivity_grid:
    sub = sensitivity_df[sensitivity_df["parameter"] == param]
    wf_span = sub["wf_sharpe"].max() - sub["wf_sharpe"].min()
    holdout_span = sub["holdout_sharpe"].max() - sub["holdout_sharpe"].min()

    best_holdout_idx = sub["holdout_sharpe"].idxmax()
    best_holdout_val = sub.loc[best_holdout_idx, "value"]

    robustness_rows.append({
        "parameter": param,
        "wf_sharpe_span": wf_span,
        "holdout_sharpe_span": holdout_span,
        "best_holdout_value": best_holdout_val,
        "base_value": base_kwargs[param],
    })

robustness_df = pd.DataFrame(robustness_rows).sort_values("holdout_sharpe_span", ascending=False)
print("Most sensitive parameters (higher span = more sensitive):")
display(robustness_df)


In [ ]:
# Save results for reuse
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = out_dir / f"volume_collapse_parameter_sensitivity_{stamp}.csv"
sensitivity_df.to_csv(out_file, index=False)

print(f"Saved sensitivity results: {out_file}")
